# TAI Quick Tutorial

Minimal end-to-end walkthrough of **Temporal Activation Injection (TAI)**:

1. Extract a per-layer τ profile for your model
2. Load the model
3. Pick a video + question
4. Inference — what does the model say about the **forward** vs. the **reversed** video, with and without TAI?

Prerequisite: `pip install -r requirements.txt`, and one video file to play with.
Step 1 needs TempCompass with `*_reverse.mp4` clips (see README);
TAI is τ-profile-guided, so the profile from step 1 drives steps 2-4.

In [ ]:
MODEL = "Qwen/Qwen2.5-VL-7B-Instruct"   # example model — swap in any Qwen-VL / InternVL / Molmo2 / Gemma4 checkpoint
N_FRAMES = 16
BETA = 0.7                               # injection strength for Qwen2.5-VL; tune per model via a TempCompass sweep
PROFILE = f"tau_norm_16f/tau_norm_{MODEL.split('/')[-1]}.json"   # produced by step 1

## 1. Extract a τ profile

Measures ‖h(V) − h(V_rev)‖ per layer over TempCompass direction pairs and writes
`tau_norm_16f/tau_norm_<model>.json` (+ plots). The JSON's peak layer becomes the
TAI source layer.

In [ ]:
TC_V = "/path/to/TempCompass/videos"     # must contain *_reverse.mp4 pairs
TC_QA = "/path/to/TempCompass/yes_no.json"

!python tau_profile/tau_norm_multi_model.py \
    --video_dir {TC_V} --qa_json {TC_QA} \
    --model_name {MODEL} --max_pairs 50 --n_frames {N_FRAMES} --output_dir tau_norm_16f

## 2. Load the model

In [ ]:
from tai.tai_core import (
    load_model_adapter, prepare_input, prepare_reverse_input,
    extract_hidden, load_tau_profile,
)

adapter = load_model_adapter(MODEL)

## 3. Pick a video + question

In [ ]:
VIDEO = "/path/to/your/video.mp4"
PROMPT = "Describe this video in one or two sentences, focusing on what changes over time."

## 4. Inference — forward vs. reversed video, baseline vs. TAI

The benchmark path (`tai_forward` in `tai/tai_core.py`) scores a single answer
token from logits. For this demo we want readable *sentences*, so the helper cell
below wraps the same TAI mechanism around greedy generation — tutorial-only code,
built from `tai_core` primitives: τ is extracted and injected during the prefill
exactly as in `tai_forward` (the `shape[1] > 1` guard keeps hooks inert on decode
steps), then decoding continues from the steered prefill via the KV cache.

A temporally-blind baseline tends to describe the forward and reversed clips the
same way; with TAI the two descriptions should diverge.

In [ ]:
import torch

@torch.no_grad()
def baseline_generate(adapter, inputs, max_new_tokens=64):
    """Greedy free-form generation. Returns decoded text."""
    if adapter.family == "internvl" and "inputs_embeds" in inputs:
        gen_ids = adapter.model.language_model.generate(
            inputs_embeds=inputs["inputs_embeds"],
            attention_mask=inputs["attention_mask"],
            max_new_tokens=max_new_tokens, do_sample=False)
        return adapter.tokenizer.batch_decode(gen_ids, skip_special_tokens=True)[0].strip()
    gen_ids = adapter.model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    in_ids = inputs.get("input_ids")
    if in_ids is not None:
        gen_ids = gen_ids[:, in_ids.shape[1]:]
    tok = adapter.processor if adapter.processor is not None else adapter.tokenizer
    return tok.batch_decode(gen_ids, skip_special_tokens=True)[0].strip()

@torch.no_grad()
def tai_generate(adapter, inputs, inputs_rev, src_layer, inject_start, beta,
                 tau_profile=None, max_new_tokens=64):
    """TAI + greedy generation (tutorial-only wrapper around tai_core primitives)."""
    layers = adapter.get_layers()
    n = len(layers)

    # (1/3) reverse-video hidden at src_layer (EarlyExit)
    h_rev = extract_hidden(adapter, inputs_rev, src_layer)

    # (2/3) injection schedule (τ-profile-guided, else linear decay)
    inject_handles = []
    def make_inject(b_val, tv):
        def hook_fn(mod, inp2, out2):
            o2 = out2[0] if isinstance(out2, tuple) else out2
            if o2.shape[1] > 1:   # prefill only — decode steps untouched
                o2[0, -1, :] += b_val * tv.to(o2.device, dtype=o2.dtype)
            return out2 if not isinstance(out2, tuple) else (o2,) + out2[1:]
        return hook_fn
    def register_inject(tau_vec, inj):
        for li in range(inj, n):
            if tau_profile is not None:
                wt = tau_profile[li] if li < len(tau_profile) else 0.0
                if wt < 1e-6: continue
                b = beta * wt
            else:
                b = beta * (1.0 - ((li - inj) / max(1, n - inj - 1)))
            inject_handles.append(layers[li].register_forward_hook(make_inject(b, tau_vec)))

    # (3/3) compute τ = h(V) − h(V_rev) once, on the prefill pass
    state = {"fired": False}
    def src_hook(module, inp, out):
        o = out[0] if isinstance(out, tuple) else out
        if state["fired"] or o.shape[1] <= 1:
            return
        state["fired"] = True
        tau_vec = o[0, -1, :].detach().clone() - h_rev.to(o.device)
        register_inject(tau_vec, inject_start if inject_start > src_layer else src_layer + 1)

    handle = layers[src_layer].register_forward_hook(src_hook)
    try:
        text = baseline_generate(adapter, inputs, max_new_tokens=max_new_tokens)
    finally:
        handle.remove()
        for h in inject_handles: h.remove()
        torch.cuda.empty_cache()
    return text

In [ ]:
import os

# TAI is τ-profile-guided: the profile peak becomes the source layer and
# its measured decay shapes the injection weights.
assert os.path.exists(PROFILE), "τ profile not found: run step 1 first"
tau_profile, src_layer = load_tau_profile(PROFILE)
inject_start = src_layer + 1

# Forward-video input (also returns the sampled frames, used to build the reversed input)
inputs_fwd, videos, vk = prepare_input(adapter, VIDEO, PROMPT, n_frames=N_FRAMES)
mk_rev = lambda: prepare_reverse_input(adapter, videos, PROMPT, vk, n_frames=N_FRAMES)

In [ ]:
# ---- Baseline ----
print("[baseline | forward video]")
print(baseline_generate(adapter, inputs_fwd), "\n")
print("[baseline | reversed video]")
print(baseline_generate(adapter, mk_rev()))

In [ ]:
# ---- TAI ----
# Forward clip: τ = h(V) − h(V_rev)
print("[TAI | forward video]")
print(tai_generate(adapter, inputs_fwd, mk_rev(), src_layer, inject_start, BETA,
                   tau_profile=tau_profile), "\n")

# Reversed clip as the main input — its temporal reverse is the original clip,
# so the roles of the two inputs simply swap.
print("[TAI | reversed video]")
print(tai_generate(adapter, mk_rev(), inputs_fwd, src_layer, inject_start, BETA,
                   tau_profile=tau_profile))

For full benchmark evaluation (TempCompass / TVBench / MVBench / AoTBench), see the
**Run evaluations** section of the README.